# 🏘️ Análisis de Participación Juvenil en la Parroquia de O Milladoiro

## Contexto y Autor

Me llamo Filipi, soy de O Milladoiro y estoy en formación como Data Scientist.
He elegido nuestra parroquia como caso de estudio real para aplicar técnicas
de análisis de datos a un problema concreto de la comunidad.

**Objetivo:** entregar un informe técnico basado en datos reales a la parroquia
de O Milladoiro, para que puedan tomar decisiones sobre cómo reconectar
con los jóvenes — no basadas en suposiciones, sino en evidencia.

---

## Fuentes de Datos

**1. Encuesta propia — "Viviendo O Milladoiro: Comunidad y Futuro"**
- 10 preguntas dirigidas a jóvenes de 18 a 35 años de O Milladoiro
- Distribuida via redes sociales y redes de comunicación locales
- Meta: 100 respuestas o más, con el apoyo del cura de la parroquia
- Variables: perfil demográfico, frecuencia de participación,
  barreras de acceso, actividades preferidas, canal digital

**2. Datos del IGE (Instituto Galego de Estatística)**
- Población de Ames por género y edad (2021–2025)
- Obtenidos via API oficial del IGE
- Objetivo: contextualizar la muestra de la encuesta con la población real

---

## Objetivo de este Notebook

En este notebook realizaremos:
1. Extracción de datos del IGE via API
2. Carga y primera inspección de los datos de la encuesta
3. Limpieza y tratamiento de problemas esperados:
   - Respuestas incompletas o vacías
   - Registros fuera del rango de edad (18–35)
   - Valores nulos o inconsistentes
4. Exportación de los datasets limpios a `data/processed/`

---

## Estructura del Proyecto

| Notebook | Contenido |
|---|---|
| `01_extraccion_y_limpieza` | Extracción IGE, carga encuesta, limpieza |
| `02_analisis_exploratorio` | Visualizaciones, patrones, insights |
| `03_modelo_ia_clustering` | K-Means para identificar perfiles de jóvenes |

**Módulos en `src/`:**
- `base_datos.py` — gestión de datos de la encuesta en SQLite
- `modelos_ia.py` — entrenamiento y evaluación del clustering

> **Resultado esperado del notebook 03:** identificar 3-4 perfiles
> de jóvenes con comportamientos distintos respecto a la parroquia,
> para proponer soluciones específicas a cada perfil.

In [1]:
# Importar librerias necesarias
import requests
import json
import sqlite3 as sql
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# Definir la ruta de este archivo actual (notebooks/01_extraccion_y_limpieza.ipynb)
BASE_DIR = Path().resolve().parent
# Rutas
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Cofiguracion
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12,6)

print(f"Librerias cargadas correctamente.")
print(f"Ruta base: {BASE_DIR}")

Librerias cargadas correctamente.
Ruta base: /Users/filipihenrique/Desktop/analisis_secularizacion_o_milladoiro


In [2]:
url = "https://www.ige.gal/igebdt/igeapi/json/datos/10611/9915:15002"
response = requests.get(url)
data = response.json()
# Convertir a DataFrame
df_ige = pd.DataFrame(data["datos"], columns=data["variables"])
display(df_ige.head(10))
print("Shape:", df_ige.shape)

,CodTempo,Tempo,Sexo,Idade,CodEspazo,Espazo,DatoN,DatoT
0,2021,2021,Total,Total,15002,15002 Ames,32062,32.062
1,2021,2021,Total,0,15002,15002 Ames,234,234
2,2021,2021,Total,1,15002,15002 Ames,274,274
3,2021,2021,Total,2,15002,15002 Ames,260,260
4,2021,2021,Total,3,15002,15002 Ames,317,317
5,2021,2021,Total,4,15002,15002 Ames,364,364
6,2021,2021,Total,5,15002,15002 Ames,349,349
7,2021,2021,Total,6,15002,15002 Ames,373,373
8,2021,2021,Total,7,15002,15002 Ames,350,350
9,2021,2021,Total,8,15002,15002 Ames,355,355


Shape: (1530, 8)


In [3]:
# Eliminar columnas CodTempo, CodEspazo, Espazo y DatoT
df_limpio = df_ige.drop(columns=["CodTempo", "CodEspazo", "Espazo", "DatoT"])
# filtrar solo las filas donde Sexo sea "Total"
df_limpio = df_limpio[df_limpio["Sexo"] == "Total"]
# Convertir columna Idade a int
df_limpio["Idade"] = pd.to_numeric(df_limpio["Idade"], errors="coerce")
# filtrar las edades entre 18 y 35
df_limpio = df_limpio[(df_limpio["Idade"] >= 18) & (df_limpio["Idade"] <=35)]
# Renombrar columnas "Tempo", "Sexo", "Idade" y "DatoN" 
df_limpio = df_limpio.rename(columns={
    "Tempo": "ano",
    "Sexo": "sexo",
    "Idade": "edad",
    "DatoN": "poblacion"
})
# guardar el DataFrame limpio en CSV
df_limpio.to_csv(DATA_PROCESSED / "poblacion_jovenes_ames.csv", index=False)
print("CSV guardado")
# Añadir una columna calculada que sume la poblacion total de jovenes 18-35 por año
df_limpio["poblacion"] = pd.to_numeric(df_limpio["poblacion"], errors="coerce")
jovenes_por_ano = df_limpio.groupby("ano")["poblacion"].sum().reset_index()
jovenes_por_ano.columns = ["ano", "total_jovenes_18_35"]
print(jovenes_por_ano)
jovenes_por_ano.to_csv(DATA_PROCESSED / "jovenes_18_35_por_año.csv", index=False)
print("CSV guardado")


CSV guardado
    ano  total_jovenes_18_35
0  2021                 5748
1  2022                 5806
2  2023                 5946
3  2024                 6036
4  2025                 6116
CSV guardado


## Universo Muestral - Jóvenes 18-35 en Ames (IGE 2024)

- **Total jóvenes 18-35 años en Ames (2024): ** 6.036 personas
- **Meta de encuesta:** 100 respuestas
- **Margen de error:** ±9.7% (con nivel de confianze del 95%)
- **Tendencia:** crecimiento sostenido - Ames ganó ~400 jóvenes en 4 años

> Estos datos contextualizan la muestra de la encuesta y confirman
> que O Milladoiro tiene una población jovén significativa y en crecimiento.

## 📝 Fase 2: Análisis de la Encuesta Personal (Datos Primarios)

Una vez analizado el contexto demográfico general de **Ames** a través del IGE, procedemos a integrar los datos recolectados directamente en **O Milladoiro**. 

Esta sección es el núcleo del proyecto, donde contrastamos la estadística oficial con la realidad percibida por los jóvenes. 

### Objetivos de esta sección:
1. **Carga y Limpieza:** Transformar las respuestas de Google Forms en un formato procesable.
2. **Representatividad:** Comparar si el perfil de nuestra muestra (30 respuestas actuales) se alinea con la realidad demográfica de la zona.
3. **Persistencia:** Volcar estos datos en nuestra base de datos SQL para futuros cruces de información.

In [4]:
# 1. Ruta al archivo de la encuesta (Usando la variable DATA_RAW)
SURVEY_PATH = DATA_RAW / "encuesta_personal.csv"

# 2. Carga inicial
df_encuesta = pd.read_csv(SURVEY_PATH)

# 3. Limpieza de nombres de columnas (Google sheets usa preguntas, los cambio por etiquetas)
columnas_nuevas = {
    "Marca temporal": "timestamp",
    "1. Edad: ______ (Respuesta corta, validada como número).": "edad",
    "2.  Sexo/Género:": "genero",
    "3. ¿A qué te dedicas principalmente?": "ocupacion",
    "4. ¿Cuánto tiempo llevas viviendo en O Milladoiro?": "antiguedad",
    "5. ¿Con qué frecuencia participas en actividades organizadas por la Parroquia de O Milladoiro?": "frecuencia",
    "6. Del 1 al 5, ¿cuánta conexión sientes con el mensaje o lenguaje que se usa en la iglesia hoy?": "conexion",
    "7. Si tuvieras que elegir UNA razón por la que no participas más, ¿cuál sería?": "barrera",
    "8. ¿Qué tipo de actividad te haría considerar acercarte más? (Elige máximo 2).": "interes", 
    "9. ¿En qué horario es más probable que asistas a algo un domingo?": "horario",
    "10. ¿Te resultaría útil una vía de contacto por WhatsApp/Instagram para conocer gente y actividades de tu edad en el pueblo?": "canal_info"
}

df_encuesta.rename(columns=columnas_nuevas, inplace=True)
# 4. Vista rapida de los primeros datos
print(f"Se han cargado {len(df_encuesta)} respuestas con éxito.")
df_encuesta.head(3)

Se han cargado 30 respuestas con éxito.


,timestamp,edad,2. Sexo/Género:,ocupacion,antiguedad,frecuencia,conexion,barrera,interes,horario,canal_info
0,23/03/2026 16:29:45,33,Mujer,• Opositor/a o en búsqueda de empleo.,• Toda la vida.,• Nunca o casi nunca,1.0,• Los temas que se tratan no me interesan,"• Deporte o excursiones (Rutas, Camino de Sant...",• Tarde (18:00 - 19:30),"• Sí, me gustaría participar"
1,23/03/2026 16:44:09,31,Mujer,• Trabajador/a.,• Menos de 2 años (Nuevo/a en el pueblo).,"• Solo en fechas señaladas (Navidad, Fiestas, ...",3.0,• Los horarios no coinciden con mi vida,"• Eventos sociales (Cenas, música en directo, ...",• Tarde (18:00 - 19:30),• Solo para recibir información puntual
2,23/03/2026 21:49:42,46,Mujer,• Trabajador/a.,Opción 3,"• Solo en fechas señaladas (Navidad, Fiestas, ...",3.0,• Los horarios no coinciden con mi vida,"• Eventos sociales (Cenas, música en directo, ...",• Mañana (11:00 - 12:30),"• Sí, me gustaría participar"


In [5]:
df_edad = df_encuesta["edad"].value_counts()
print(df_edad)

edad
33    3
34    3
35    3
31    2
46    2
28    2
37    2
42    2
40    2
50    1
24    1
49    1
36    1
30    1
43    1
38    1
39    1
52    1
Name: count, dtype: int64


### ⚠️ Nota sobre el filtrado de la muestra
Durante la fase de recolección, se han obtenido respuestas de personas fuera del rango objetivo (18-35 años), llegando hasta los 52 años.

Para mantener la integridad del estudio de **Secularización Juvenil**, procederemos a:
1. **Segmentar la muestra:** Separar el "Grupo Objetivo" (18-35) del "Grupo Adulto" (36+).
2. **Análisis Comparativo:** No descartaremos los datos de >35, los usaremos para contrastar si las barreras de participación son la mismas entre jóvenes y adultos.

In [6]:
# 1. Aseguramos que la columna "edad" sea númerica.
df_encuesta["edad"] = pd.to_numeric(df_encuesta["edad"], errors="coerce")

# 2. Creamos un nuevo DataFrame especifico para el estudio juvenil (18-35).
df_jovenes = df_encuesta[(df_encuesta["edad"] >= 18) & (df_encuesta["edad"] <= 35)].copy()

# 3. Creamos el Dataframe de contraste (Adultos).
df_adultos = df_encuesta[(df_encuesta["edad"] > 35)].copy()

print(f"✅ Segmentación finalizada:")
print(f"- Jóvenes (Objetivo): {len(df_jovenes)} personas")
print(f"- Adultos (Contraste): {len(df_adultos)} personas")

✅ Segmentación finalizada:
- Jóvenes (Objetivo): 15 personas
- Adultos (Contraste): 15 personas


### 💡 Hipótesis de Contraste Intergeneracional

Al contar con una muestra extendida hasta los 52 años, se abre una oportunidad analítica superior: **¿Es la desafección religiosa una cuestión de edad o es transversal a la comunidad?**

#### **Planteamiento:**
* **Segmento Adulto (36-52 años):** Se plantea la hipótesis de que su baja participación se debe a factores logísticos o estructurales (ej. *falta de tiempo*, *cargas familiares*).
* **Segmento Joven (18-35 años):** Se plantea la hipótesis de que su baja participación se debe a factores ideológicos o de identidad (ej. *falta de conexión con el mensaje*, *desconexión emocional*).

> **Valor del análisis:** Si validamos esta diferencia, la solución para la parroquia no será solo "cambiar el horario", sino "cambiar el lenguaje" para los más jóvenes.

In [7]:
# 1. Clasificación por grupos generacionales (Feature Enginering básico)
def categorizar_generacion(edad):
    if edad <= 25:
        return "Gen Z (18-25)"
    elif edad <= 35:
        return "Millenials (26-35)"
    else:
        return "Adultos (36+)"
    
df_encuesta["grupo_etario"] = df_encuesta["edad"].apply(categorizar_generacion)

# 2. Guardar el resultado limpio para los siguientes notebooks
# Lo guardamos en "processed" para no tocar el original de "raw"
df_encuesta.to_csv(DATA_PROCESSED / "encuesta_segmentada.csv", index=False)

print("✅ Archivo 'encuesta_segmentada.csv' creado con éxito.")
print(df_encuesta["grupo_etario"].value_counts())

✅ Archivo 'encuesta_segmentada.csv' creado con éxito.
grupo_etario
Adultos (36+)         15
Millenials (26-35)    14
Gen Z (18-25)          1
Name: count, dtype: int64
